# Contextual Compression (LangChain)

Contextual Compression retrieves documents first, then shrinks them by extracting only the parts relevant to the query.

**Step 1: Install Dependencies**

In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-openai langchain-chroma
!pip install langchain==0.2.16 langchain-core==0.2.38 langchain-community==0.2.16 langchain-openai langchain-chroma


**Step 2: Imports & API Config**
- ChatOpenAI → to understand the query & compress text

- OpenAIEmbeddings → convert docs to vectors

- Chroma → vector DB

- ContextualCompressionRetriever → wrapper retriever

- LLMChainExtractor → does the actual compression

In [ ]:
import os

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

from langchain_core.documents import Document

from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor


In [ ]:
os.environ["OPENAI_API_KEY"] = "Your_api_key"
os.environ["OPENAI_API_BASE"] = "https://apidev.navigatelabsai.com"


**Step 3: Documents**

Sets your API key + backend URL.

LangChain uses these to talk to the model provider.

In [ ]:
docs=[
    Document(
        page_content="Scientists clone dinosaurs, chaos follows.",
        metadata={"year": 1993, "rating": 7.7, "genre": "sci-fi"}
    ),
    Document(
        page_content="A dream within a dream heist.",
        metadata={"year": 2010, "rating": 8.2, "genre": "sci-fi"}
    ),
    Document(
        page_content="Toys come alive when humans are away.",
        metadata={"year": 1995, "rating": 8.3, "genre": "animated"}
    ),
    Document(
        page_content="Detectives hunt a serial killer.",
        metadata={"year": 1995, "rating": 8.6, "genre": "crime"}
    ),
]


**Step 4: Embeddings + Vector Store**

Creates sample movie descriptions + metadata.
This is the knowledge base we’ll search in.

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    base_url="https://apidev.navigatelabsai.com"
)

vectorstore = Chroma.from_documents(docs, embeddings)


**Step 5: Base Retriever**

Turns each document into vectors and stores them in Chroma.

Vectors allow semantic search, not keyword search.

In [ ]:
base_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)


**Step 6: Create the Compressor**

Creates a normal retriever over the vector DB.

This finds the most similar documents for a query.

In [ ]:
llm = ChatOpenAI(
    model="gemini-2.5-flash",
    base_url="https://apidev.navigatelabsai.com"
)

compressor = LLMChainExtractor.from_llm(llm)


**Step 7: Contextual Compression Retriever**

Creates an LLM object and wraps it in LLMChainExtractor.

This is the engine that removes irrelevant text from retrieved docs.

In [ ]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)


**Step 8: Run Contextual Compression**

Combines:

Retriever (Step 6)

Compressor (Step 7)

Why:
Now retrieval automatically includes compression.

In [ ]:
query = "sci-fi movie about dinosaurs"

results = compression_retriever.get_relevant_documents(query)

print("\n--- CONTEXTUAL COMPRESSION RESULTS ---")
for r in results:
    print(r.page_content)
    print("Metadata:", r.metadata)


**Summary**
- Contextual Compression Flow

- Retrieve top-k documents from vector DB

- For each doc → pass to LLM

- LLM removes irrelevant text

- Keeps only query-related sentences

- Returns shorter, focused docs